In [ ]:
# Demonstrate the zone-based average fare per mile function
from ai_coding_tools.fare_calculator import add_avg_fare_per_mile_by_zones

# Load a sample of data for demonstration  
zone_demo_df = spark.read.table("samples.nyctaxi.trips").limit(500)

print("Sample data before adding zone-based averages:")
zone_demo_df.select("pickup_zip", "dropoff_zip", "fare_amount", "trip_distance").show(5)

# Add zone-based average fare per mile
df_with_zone_avg = add_avg_fare_per_mile_by_zones(zone_demo_df)

print("\nData with zone-based average fare per mile:")
df_with_zone_avg.select("pickup_zip", "dropoff_zip", "fare_amount", "trip_distance", "avg_fare_per_mile_by_zones").show(10)

# Show some route-specific statistics
print("\nTop 10 pickup/dropoff zone combinations by average fare per mile:")
route_stats = df_with_zone_avg.groupBy("pickup_zip", "dropoff_zip", "avg_fare_per_mile_by_zones") \
    .count() \
    .filter(F.col("avg_fare_per_mile_by_zones").isNotNull()) \
    .orderBy(F.desc("avg_fare_per_mile_by_zones")) \
    .limit(10)

route_stats.show()

print(f"\nTotal routes with zone-based averages: {route_stats.count()}")


In [ ]:
# Write the zone-based data to a table
df_with_zone_avg.write \
  .mode("overwrite") \
  .saveAsTable("main.dustinvannoy_dev.trips_with_zone_avg_fare_per_mile")

# Verify the table was created
record_count = spark.sql("SELECT COUNT(*) as total_records FROM main.dustinvannoy_dev.trips_with_zone_avg_fare_per_mile").collect()[0]['total_records']
print(f"Data successfully written to main.dustinvannoy_dev.trips_with_zone_avg_fare_per_mile")
print(f"Total records: {record_count:,}")
